[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllInVaders/aistudio-full-course/blob/main/notebooks/02_Prompts_Structured_Outputs_Gemini_Image_and_Omni_Video.ipynb)

# Module 02: System Instructions, Thinking Budgets, Structured Outputs, Gemini 3.1 Flash Image (Nano Banana 2) & Gemini Omni 1.1 Flash
### Módulo 02: Instrucciones de Sistema, Presupuesto de Razonamiento, Salidas Estructuradas, Gemini 3.1 Flash Image (Nano Banana 2) y Gemini Omni 1.1 Flash

**English Overview**: Build **Stage 1 of the Flagship Project (AI Product Studio Creative Engine)**. You will generate guaranteed Pydantic JSON launch kits with Gemini 3.7 Flash, render photorealistic studio product hero shots with **Gemini 3.1 Flash Image (`gemini-3.1-flash-image` / Nano Banana 2)**, and generate cinematic motion clips with **Gemini Omni 1.1 Flash (`gemini-omni-1.1-flash`)**.

**Resumen en Español**: Construye la **Etapa 1 del Proyecto Insignia (Motor Creativo de Estudio de Productos IA)**. Generarás kits de lanzamiento JSON validados con Pydantic usando Gemini 3.7 Flash, fotografías de estudio fotorrealistas con **Gemini 3.1 Flash Image (`gemini-3.1-flash-image` / Nano Banana 2)** y clips de video cinematográficos con **Gemini Omni 1.1 Flash (`gemini-omni-1.1-flash`)**.

In [ ]:
%pip install -q -U google-genai pydantic pillow

In [ ]:
import json
from typing import List
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

class ChannelAdCopy(BaseModel):
    channel: str
    headline_en: str
    headline_es: str
    body_copy_en: str
    body_copy_es: str

class ProductLaunchKit(BaseModel):
    product_name: str
    tagline_en: str
    tagline_es: str
    imagen_hero_prompt: str = Field(description='Detailed studio photography prompt for Gemini 3.1 Flash Image (`gemini-3.1-flash-image` / Nano Banana 2).')
    veo_teaser_prompt: str = Field(description='Cinematic motion prompt for Gemini Omni 1.1 Flash (`gemini-omni-1.1-flash`).')
    ad_campaigns: List[ChannelAdCopy]

client = genai.Client()

In [ ]:
concept = 'AeroBrew Nano: Pocket-sized ultrasonic cold-brew espresso maker for frequent flyers'

response = client.models.generate_content(
    model='gemini-3.7-flash',
    contents=f'Create a complete bilingual launch kit for: {concept}',
    config=types.GenerateContentConfig(
        system_instruction='You are an Executive Creative Director at an AI Product Studio.',
        temperature=0.4,
        response_mime_type='application/json',
        response_schema=ProductLaunchKit,
        thinking_config=types.ThinkingConfig(thinking_budget=1024),
    ),
)

launch_kit = ProductLaunchKit.model_validate_json(response.text)
print(json.dumps(launch_kit.model_dump(), indent=2, ensure_ascii=False))

In [ ]:
# Render Studio Hero Shot with Gemini 3.1 Flash Image (`gemini-3.1-flash-image` / Nano Banana 2)
from io import BytesIO
from PIL import Image

img_result = client.models.generate_content(
    model='gemini-3.1-flash-image',
    prompt=launch_kit.imagen_hero_prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio='16:9',
        output_mime_type='image/jpeg',
    ),
)

hero_image = Image.open(BytesIO(img_result.generated_images[0].image.image_bytes))
display(hero_image)